# Build circuits in Go, simulate in Python

## Prerequisites

* Python 3.10+ with this branch's built `qdk`, `ipykernel`, and IPython in the selected Python kernel, plus JupyterLab or VS Code's Python/Jupyter extensions.
* A checkout of this branch, Rust/Cargo, Go 1.20+ with cgo support, and a GCC/Clang-compatible C compiler (`CC` may select it). The QDK Go module is loaded from the locally staged native SDK; no external module downloads are required.
* The matching native circuit library/header, built from the repository root with `python source/circuit_capi/build.py`. Follow the [setup guide](README.md) first.
* macOS or Linux; use WSL2 for this native sample on Windows.
* Optional GPU simulation requires a QDK-supported GPU and drivers; it is disabled by default.

This Python-kernel notebook compiles a standalone Go client. Go constructs and owns the circuits through cgo, without embedding Python. Python later loads the exported snapshot files and simulates their QIR. There is no direct circuit simulation method in the native API.

In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
from IPython.display import Markdown, display
from qdk import Result, TargetProfile
from qdk.circuit import QuantumCircuit
from qdk.simulation import run_qir

root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "source/circuit_capi/includes/qdk_circuit.h").is_file()
    ),
    None,
)
if root is None:
    raise RuntimeError("Start this kernel inside the QDK checkout.")
sys.path.insert(0, str(root / "samples/notebooks/circuit_api"))
from _native import SOURCES, build_and_run

RUN_GPU = False
SHOTS = 128

## Clone a shared prefix into two programs

[The Go client](native/circuits.go) constructs a two-qubit H prefix and clones it into independent owners. It frees the original prefix, then extends one clone with CX and terminal measurements (Base) and the other with measurement-driven X (Adaptive). This supports constructing a family of circuits from common preparation steps.

Go controls every construction step using the [reusable Go binding](../../../source/circuit_capi/bindings/go/README.md). Its [shim.go](../../../source/circuit_capi/bindings/go/shim.go) hides cgo handles, diagnostics, and native byte buffers. Methods return ordinary Go errors. `IfOne` scopes a `Body` to a callback, finishes or cancels it, and releases dependent handles automatically. Circuits use explicit `Close` calls, with deferred cleanup errors propagated by `errors.Join`.

Edit [circuits.go](native/circuits.go), then rerun the compile cell to try another circuit. The build helper creates a temporary application module with a local replacement for the SDK's staged Go package; the main program imports that package rather than `C`.

In [ ]:
source = (SOURCES / "circuits.go").read_text(encoding="utf-8")
display(Markdown(f"```go\n{source}\n```"))

In [ ]:
work = TemporaryDirectory(prefix="qdk-go-circuits-")
directory = Path(work.name).resolve()
print("Artifact directory:", directory)
build_and_run("go", directory)
print("Native artifacts (removed by the final cleanup cell):")
for path in sorted(directory.iterdir()):
    print(path)

## Load both programs in Python and simulate on the CPU

Binary `.qdk` snapshots preserve the native circuit, so Python does not need a reconstruction script. Each has a readable `.sexpr` snapshot alongside it; loading either format restores the same circuit. They are typed QDK snapshots, not JSON or HUGR documents. The `.ll` files demonstrate native QIR compilation too; below we verify parity with compilation after loading in Python.

Both circuits produce `00` or `11`, but only the Bell circuit uses entanglement. The adaptive circuit uses a measured classical result to control X. Requesting the CPU explicitly avoids automatic backend selection.

In [ ]:
programs = {}
circuits = {}
for name, profile in [
    ("base", TargetProfile.Base),
    ("adaptive", TargetProfile.Adaptive),
]:
    payload = (directory / f"{name}.qdk").read_bytes()
    circuits[name] = QuantumCircuit.from_bytes(payload)
    assert circuits[name].to_bytes() == payload
    text = (directory / f"{name}.sexpr").read_text(encoding="utf-8")
    assert QuantumCircuit.from_text(text).to_bytes() == payload
    programs[name] = circuits[name].to_qir(profile)
    assert programs[name] == (directory / f"{name}.ll").read_text(encoding="utf-8")
    print(name, "\n", circuits[name])


def correlated_counts(samples):
    assert len(samples) == SHOTS
    assert all(
        len(s) == 2 and s[0] == s[1] and s[0] in (Result.Zero, Result.One)
        for s in samples
    )
    return Counter(
        "".join("1" if bit == Result.One else "0" for bit in s) for s in samples
    )


for name, qir in programs.items():
    samples = run_qir(qir, shots=SHOTS, seed=42, type="cpu")
    print(name, "CPU:", correlated_counts(samples))

## Opt in to the GPU

Set `RUN_GPU = True` in the setup cell and rerun this cell after loading the programs. GPU errors are surfaced without CPU fallback. The same seed need not give the same shots on different backends. Two qubits illustrate the API, not a performance advantage.

In [ ]:
if RUN_GPU:
    for name, qir in programs.items():
        samples = run_qir(qir, shots=SHOTS, seed=42, type="gpu")
        print(name, "GPU:", correlated_counts(samples))
else:
    print("GPU execution disabled; set RUN_GPU = True to enable it.")

## Inspect and exchange the Go-built circuit

After loading, all Python circuit tooling is available: inspect the typed S-expression, lower to the graph, or export OpenQASM for another tool. Graph verification returns an empty diagnostic list for a valid graph. Use the binary snapshot for exact persistence; OpenQASM is an interchange representation.

In [ ]:
graph = circuits["base"].to_graph()
assert graph.diagnostics() == []
print("Graph operations:", Counter(node.operation for node in graph.nodes))
text = circuits["adaptive"].to_text()
assert QuantumCircuit.from_text(text).to_bytes() == circuits["adaptive"].to_bytes()
print("Adaptive snapshot preview:", text[:250])
print(circuits["base"].to_qasm())

## Cleanup

Copy any snapshots or QIR you want to retain outside `directory` first. The next cell removes this notebook's temporary sources, executable, and output files, not the originals in the checkout.

In [ ]:
work.cleanup()
assert not directory.exists()